# Tutorial 1: Radon Transform, FBP, Filters and Artifacts
---
Faculty of Mathematics, Vienna, March 18, 2026.

---

### Learning Objectives
In this notebook, you will:
1. Use the scikit-image package to implement the Radon transform and Filtered Back-Projection
2. Explore the properties of the Radon transform
3. Explore image filtering and limited angle tomography

---
## 1. Setup and Imports

First, let's import the necessary packages and modules.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon, iradon, rescale
from skimage.data import shepp_logan_phantom

print("Libraries imported successfully!")

## 2. Mathematical Background: The Radon Transform

### Definition
The **Radon transform** $\mathcal{R}$ of a (sufficiently regular) function $f(x,y)$ defined on $\mathbb{R}^2$ computes line integrals along different angles:

$$\mathcal{R}[f](t, \theta) \;=\; \int_{-\infty}^{\infty} f(t\cos\theta - s\sin\theta, t\sin\theta+s\cos\theta) \, ds$$

where:
- $t$ is the signed perpendicular distance from the origin to the line
- $\theta$ is the angle of the line with respect to the x-axis

In CT scanning:
- $f$ represents the attenuatiopn coefficient in a cross-section of a sample
- Each line integral represents the total attenuation of an X-ray beam passing through the object (2D case)
- By rotating the X-ray source around the object, we collect projections at different angles
- The collection of all projections is called a **sinogram**

---
## 3. Implementation of a phantom & its Radon Transform

As our original image, we will use the socalled Shepp-Logan phantom, a standard test object in CT, whose structure is determined mathematically (ellipses). 
We have already imported it from the module `skimage.data`.

In [ ]:
# create the phantom
image = shepp_logan_phantom() # (400, 400) float64 image

print(f"Image shape: {image.shape}; Image type: {type(image)}; Image Dimensions: {image.ndim}")

In [ ]:
# we rescale it 
image = rescale(image, scale=0.3, mode='reflect', channel_axis=None) 
# scale 0.4 -> keep 40% of the pixels
# mode --> what to do at the border; 'reflect' in CT helps reducing artifacts
# channel_axis=None --> image is grayscale
print(f"Image shape: {image.shape}; Image type: {type(image)}; Image Dimensions: {image.ndim}")

In [ ]:
# plot the phantom
fig = plt.figure(1)
plt.imshow(image, cmap='gray')
plt.title('Shepp-Logan phantom')
plt.axis('off')
plt.show()

We now introduce a 1D array of angles using the numpy function `linspace`.

**Question:** How many projection angles?
As a rule, the number of projections should be about the same as the number of pixels there are across the object

In [ ]:
# number of projection angles
num_angles = max(image.shape) 
# theta: array with equally spaced angles in degree
theta = np.linspace(0.0, 180.0, num=num_angles, endpoint=False)
print(f"Number of projection angles: {len(theta)}")
print(f"Angular spacing: {theta[1] - theta[0]:.2f} degrees")

We create the sinogram using the function `radon` from the module `skimage.transform`.

In [ ]:
# Create the sinogram
sinogram = radon(image, theta=theta)
# Info about the Sinogram
print(f"Sinogram dimensions: {sinogram.ndim}")
print(f"Rows (detector positions): {sinogram.shape[0]}")
print(f"Columns (projection angles): {sinogram.shape[1]}")

In [ ]:
# To check the information about the function used, you can uncomment the following cell
# help(radon)

In [ ]:
fig, ax1 = plt.subplots()

im = ax1.imshow(
    sinogram,
    cmap='gray',
    aspect='auto',
    origin='lower',      # detector index increases upward
    extent=[theta[0], theta[-1], 0, sinogram.shape[0] - 1]  # x: degrees, y: detector index
)

ax1.set_title("Radon transform\n(Sinogram)")
ax1.set_xlabel("Projection angle (deg)")
ax1.set_ylabel("Projection position (pixels)")

fig.tight_layout()
plt.show()

## 4. Reconstruction with the Back-Projection

The Back-projection was a "naive" reconstruction method that was introduced to invert the Radon transform. 
The idea is to calculate the average of the function with respect to the angles:

$$\mathcal{R}^{\sharp}g[x,y] = \frac{1}{\pi}\int_0^{\pi} g(\theta,x\cos\theta + y\sin\theta) d\theta.$$

Unfortunately, it doesn't work very well, let us see why in the following code!

In [ ]:
reconstruction_bp = iradon(sinogram, theta=theta, filter_name=None, circle=False, output_size=image.shape[0])

print(f"Reconstructed image shape: {reconstruction_bp.shape}")

In [ ]:
# Visualize original vs reconstruction
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Original
ax1.imshow(image, cmap='gray')
ax1.set_title('Original Phantom', fontsize=14)
ax1.axis('off')

# Sinogram
ax2.imshow(sinogram, cmap='gray', aspect='auto', origin='lower', 
           extent=[theta[0], theta[-1], 0, sinogram.shape[0] - 1])
ax2.set_title('Sinogram (Radon Transform)', fontsize=14)
ax2.set_xlabel('Projection angle (deg)', fontsize=12)
ax2.set_ylabel('Detector position (pixels)')

# Reconstruction
ax3.imshow(reconstruction_bp, cmap='gray')
ax3.set_title('Reconstructed Image (Backprojection)', fontsize=14)
ax3.axis('off')
plt.tight_layout()
plt.show()

## 5. Reconstruction using the Filtered Back Projection (FBP)

The Back-Projection is not a good operator to invert the Radon Transform, since it averages the intensities values and it smooths the edges. A solution is the **filtered backprojection** formula.

In Python the corresponding function is named `iradon`. The only parameter that one can change is the `filter_name`. Let us explore it.

### Common Filters:
- **Ramp**: (`filter_name='ramp'`) 
- **Shepp-Logan**: (`filter_name='shepp-logan'`) 
- **Cosine**: (`filter_name='cosine'`) 
- **Hamming**: (`filter_name='hamming'`) 
- **Hann**: (`filter_name='hann'`) 

In [ ]:
# Reconstruct the image using FBP and a ramp filter
reconstruction_fbp = iradon(sinogram, theta=theta, filter_name='ramp', circle=True)

# Visualize original vs reconstruction
fig, (plt1, plt2, plt3) = plt.subplots(1, 3, figsize=(18, 5))

# Original
plt1.imshow(image, cmap='gray')
plt1.set_title('Original Phantom', fontsize=14)
plt1.axis('off')

# Sinogram
plt2.imshow(sinogram, cmap='gray', origin='lower', 
           extent=[theta[0], theta[-1], 0, sinogram.shape[0] - 1])
plt2.set_title('Sinogram (Radon Transform)', fontsize=14)
plt2.set_xlabel('Projection angle', fontsize=12)

# Reconstruction
plt3.imshow(reconstruction_fbp, cmap='gray')
plt3.set_title('Reconstructed Image (Filtered Backprojection)', fontsize=14)
plt3.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Here we print the graph of the filters
from skimage.transform.radon_transform import _get_fourier_filter

filters = ['ramp', 'shepp-logan', 'cosine', 'hamming', 'hann']

for ix, f in enumerate(filters):
    response = _get_fourier_filter(2000, f)
    plt.plot(response, label=f)

plt.xlim([0, 1000])
plt.xlabel('frequency')
plt.legend()
plt.show()

## Measuring error
To measure how well the reconstruction has worked, we can use the RMSE (Root Mean Square Error):
$$\text{RMSE} = \sqrt{\frac{1}{N_r\cdot N_c}\sum_{i=1}^{N_r} \sum_{j=1}^{N_c} (I[i,j] - R[i,j])^2}$$
where
- $I$ represents the original image, the Shepp-Logan phantom;
- $R$ represents the reconstructed image using different filters.

We apply this metric to compare different reconstructions of the same image.

A lower RMSE indicates that the reconstruction is similar to the original image.

In [ ]:
# Calculate RMSE
def RMSE(image,reconstruction):
    return np.sqrt(np.mean((reconstruction - image)**2))

error = RMSE(image,reconstruction_fbp)

print(f"Root Mean Square Error (RMSE): {error:.3g}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4.5))
ax1.set_title("Reconstruction\nFiltered back projection")
ax1.imshow(reconstruction_fbp, cmap='gray')
ax2.set_title("Reconstruction error in FBP")
ax2.imshow(reconstruction_fbp - image, cmap='gray', vmin=-0.2, vmax=0.2)
plt.show()

## Exercise 1: Phantoms, sinograms and FBP

### 🧪 Exercise 1a: The sinogram of a small dot

**Task**: Create here the code for the sinogram of a small white dot (a circular disc with small radius) to be set in the upper left part of the image. 
Print the phantom and the sinogram.

**Questions**:
1. Can you see where the name sinogram comes from?
2. Create a second phantom with a small white dot set in the centre of the image. Compare the two sinograms.

**Time: 10 min**


---

In [ ]:
# Write your code here
sizeDot = 160
phantomDot = np.zeros((sizeDot,sizeDot))
# ...

### Exercise 1b: Radon transform of a series of phantoms
1. Create a phantom  composed by a small square of side 10 in the centre and attenuation coefficient 0.8 on a black background (256,256)
2. Create another phantom (256,256) composed by four discs of radius 10 with attenuation coefficient 0.5: one in the upper left part of the image, one upper right part of the image, one in the lower left part and one in the lower right part. Its centre should stay at distance 1/4 from the sides of the image.
3. Compute the Radon transform and the FBP of the two phantoms separately
4. Create a unique phantom composed by two above phantoms. Compute its Radon transform and the FBP
5. Compare the three sinograms: what can you observe? 



**Time: 15-20 minutes**

In [ ]:
# Write your code here
size1 = 256
phantom1 = np.zeros((size1,size1))
# ...

## 6. X-ray tomography with limited data - A first glance at tomographic artifacts

### Sparse full-angle tomography

In many practical tomographic applications, there are often restrictions on the number of available projection directions or an angle of view. 

With full-data tomography, the filtered-back-projection type algorithms are still efficient and used nowadays in CT machines. This is no longer true when dealing with **sparse full-angle tomography**. In this case, the projections are taken from full angle but with large uniform angular steps. 

Let's explore how this affects reconstruction quality.

In [ ]:
# Compare reconstructions with different numbers of projections
num_projections_list = [10, 30, 60, 180]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, num_proj in enumerate(num_projections_list):
    
    # Create theta array with sparse full-data projections
    theta_limited = np.linspace(0., 180., num_proj, endpoint=False)
    
    # Compute Radon transform with sparse angles
    sinogram_limited = radon(image, theta=theta_limited)
    
    # Reconstruction step
    reconstruction_limited = iradon(sinogram_limited, theta=theta_limited, 
                                    filter_name='ramp')
    
    # Calculate error
    error_limited = RMSE(image,reconstruction_limited)
    
    # Plot sinogram
    axes[0, i].imshow(sinogram_limited, cmap='gray', aspect='auto')
    axes[0, i].set_title(f'Sinogram: {num_proj} angles', fontsize=12)
    axes[0, i].axis('off')
    
    # Plot reconstruction
    axes[1, i].imshow(reconstruction_limited, cmap='gray')
    axes[1, i].set_title(f'Reconstruction: {num_proj} angles\nRMSE: {error_limited:.4f}', 
                          fontsize=12)
    axes[1, i].axis('off')

plt.suptitle('Sparse Full-Angle Tomography', fontsize=12)
plt.tight_layout()
plt.show()

We can see **streaking artifacts** radiating from high-contrast edges in the case of fewer projections.

**Question**:
Consider the RMSE of the reconstructions with number of projections 30, 60, 120, 240. Does doubling the number of projections always halve the error?

### Limited-angle tomography

In this case the projection images are available only from a restricted angle of view. Typical artifacts are **blurring** and **elongation**

Let us explore this in the following cells. 

In [ ]:
# Load the classic Shepp-Logan CT phantom and rescale it
phantom_big = shepp_logan_phantom()          # 400x400
phantom_resc = rescale(phantom_big, scale=0.4, mode='reflect')

# Define the vector of angles theta
angles = np.linspace(0, 120, 24, endpoint=False)
sinogram_la = radon(phantom_resc, theta=angles, circle=False)

# Reconstruct the image using filtered-back-projection
reconstruction_fbp = iradon(sinogram_la, theta=angles, filter_name='ramp', circle=False)
rmse_fbp = RMSE(phantom_resc, reconstruction_fbp)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(phantom_resc, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Ground truth')
axes[0].axis('off')
axes[1].imshow(reconstruction_fbp, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'FBP reconstruction\nRMSE = {rmse_fbp:.4g}')
axes[1].axis('off')
axes[2].imshow(reconstruction_fbp - phantom_resc, cmap='gray')
axes[2].set_title('Error')
axes[2].axis('off')
plt.suptitle('Limited Angle Tomography', fontsize=11)
plt.tight_layout()
plt.show()


## End for today!!

## Extra Exercises

### Exercise 1: Radon transform of nested circles
Create four concentric discs with different attenuation coefficients (image of size (256,256)). How does the sinogram looks like? What happens if you print the projection profile at row 128?

In [ ]:
# Write your code here

### Exercise 2: Filters

1. Write a function that takes as input a sinogram and a filter and returns the reconstructed image and the RMSE.
2. Use this function to compute the FBP reconstructions of the given noisy sinogram using different type of filters
3. Plot the reconstructed images. In the title, add the name of the filter used and the value of the RMSE.

**Questions**
- How does the RMSE change with the different filters?
- Does the RMSE reflect the different outcomes from the image?

In [ ]:
# Write your code here
from skimage.util import random_noise

image = shepp_logan_phantom()
image = rescale(image, scale=0.4, mode='reflect', channel_axis=None)
theta = np.linspace(0.0, 180.0, max(image.shape), endpoint=False)
sigma = 1 # noise level
sinogram_noisy = random_noise(sinogram, mode='gaussian', var=sigma**2, clip=False)

# ...

---
### References
- T. Feeman: "The Mathematics of Medical Imaging"
- J. L. Mueller, S. Siltanen:"Linear and Nonlinear Inverse Problems with Practical Applications"
- scikit-image documentation: https://scikit-image.org/docs/stable/api/skimage.transform.html
---